# StockGen Upscaler — Full Model Benchmark
ทดสอบทุกโมเดล (x2plus, x4plus, ultrasharp, anime, v3) + GFPGAN + Benchmark

In [ ]:
# 1) Install dependencies
import sys, os, types, time
!pip install -q realesrgan gfpgan opencv-python-headless matplotlib pillow

import torchvision.transforms.functional as F_tv
shim = types.ModuleType('torchvision.transforms.functional_tensor')
shim.rgb_to_grayscale = F_tv.rgb_to_grayscale
sys.modules['torchvision.transforms.functional_tensor'] = shim
print('Dependencies installed')

In [ ]:
# 2) Imports
import torch
import urllib.request
from basicsr.archs.rrdbnet_arch import RRDBNet
from realesrgan import RealESRGANer
from realesrgan.archs.srvgg_arch import SRVGGNetCompact
from gfpgan import GFPGANer
import cv2, base64, tempfile, numpy as np, time, matplotlib.pyplot as plt
from PIL import Image
import io

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# 3) Model configurations — ALL models

def convert_state_dict(state_dict, num_block):
    if not any(k.startswith('model.0.') for k in state_dict.keys()):
        return state_dict
    print('Converting classic ESRGAN keys to RRDBNet...')
    new_sd = {}
    for k, v in state_dict.items():
        if k.startswith('model.0.'):
            new_sd[k.replace('model.0.', 'conv_first.')] = v
        elif k.startswith('model.1.sub.'):
            parts = k.split('.')
            try:
                block_idx = int(parts[3])
                if block_idx == num_block:
                    new_sd[k.replace(f'model.1.sub.{block_idx}.', 'conv_body.')] = v
                else:
                    nk = k.replace('model.1.sub.', 'body.')
                    nk = nk.replace('.RDB', '.rdb')
                    for i in range(1, 6):
                        nk = nk.replace(f'.conv{i}.0.', f'.conv{i}.')
                    new_sd[nk] = v
            except:
                new_sd[k] = v
        elif k.startswith('model.3.'):
            new_sd[k.replace('model.3.', 'conv_up1.')] = v
        elif k.startswith('model.6.'):
            new_sd[k.replace('model.6.', 'conv_up2.')] = v
        elif k.startswith('model.8.'):
            new_sd[k.replace('model.8.', 'conv_hr.')] = v
        elif k.startswith('model.10.'):
            new_sd[k.replace('model.10.', 'conv_last.')] = v
        else:
            new_sd[k] = v
    return new_sd

RRDB_MODELS = {
    'x2plus': {'url': 'https://huggingface.co/nateraw/real-esrgan/resolve/main/RealESRGAN_x2plus.pth',
               'file': 'RealESRGAN_x2plus.pth', 'scale': 2, 'block': 23, 'tier': 'normal'},
    'x4plus': {'url': 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth',
               'file': 'RealESRGAN_x4plus.pth', 'scale': 4, 'block': 23, 'tier': 'normal'},
    'anime': {'url': 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.2.4/RealESRGAN_x4plus_anime_6B.pth',
              'file': 'RealESRGAN_x4plus_anime_6B.pth', 'scale': 4, 'block': 6, 'tier': 'normal'},
    'ultrasharp': {'url': 'https://huggingface.co/Kim2091/UltraSharp/resolve/main/4x-UltraSharp.pth',
                   'file': '4x-UltraSharp.pth', 'scale': 4, 'block': 23, 'tier': 'ultra'},
}

# === v3 Tiny Model (SRVGGNetCompact architecture, NOT RRDBNet) ===
V3_CFG = {
    'url': 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/realesr-general-x4v3.pth',
    'file': 'realesr-general-x4v3.pth',
    'scale': 4,
    'tier': 'general',
}

weights_dir = './weights'
os.makedirs(weights_dir, exist_ok=True)
print(f'All {len(RRDB_MODELS)} RRDB models + v3 tiny model configured')

In [ ]:
# 4) Model loader functions

def get_rrdb_upsampler(model_key, use_half=True):
    """Load RRDBNet-based model (x2plus, x4plus, ultrasharp, anime)"""
    cfg = RRDB_MODELS[model_key]
    model_path = os.path.join(weights_dir, cfg['file'])
    if not os.path.exists(model_path):
        print(f'Downloading {cfg["file"]}...')
        urllib.request.urlretrieve(cfg['url'], model_path)
    try:
        loadnet = torch.load(model_path, map_location='cpu', weights_only=True)
    except:
        loadnet = torch.load(model_path, map_location='cpu', weights_only=False)
    if 'params' in loadnet:
        state_dict = loadnet['params']
    elif 'params_ema' in loadnet:
        state_dict = loadnet['params_ema']
    else:
        state_dict = loadnet
    # Convert classic keys if needed
    converted = convert_state_dict(state_dict, cfg['block'])
    torch.save({'params': converted}, model_path)
    
    model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64,
                    num_block=cfg['block'], num_grow_ch=32, scale=cfg['scale'])
    return RealESRGANer(scale=cfg['scale'], model_path=model_path,
                        model=model, tile=800, tile_pad=10, pre_pad=10,
                        half=use_half and device.type == 'cuda', device=device), cfg['scale']

def get_v3_upsampler(use_half=True):
    """Load v3 tiny model (SRVGGNetCompact)"""
    model_path = os.path.join(weights_dir, V3_CFG['file'])
    if not os.path.exists(model_path):
        print(f'Downloading {V3_CFG["file"]}...')
        urllib.request.urlretrieve(V3_CFG['url'], model_path)
    
    model = SRVGGNetCompact(num_in_ch=3, num_out_ch=3, num_feat=64,
                            num_conv=32, upscale=4, act_type='prelu')
    return RealESRGANer(scale=4, model_path=model_path,
                        model=model, tile=800, tile_pad=10, pre_pad=10,
                        half=use_half and device.type == 'cuda', device=device), 4

def get_face_enhancer(upsampler, scale):
    path = os.path.join(weights_dir, 'GFPGANv1.3.pth')
    if not os.path.exists(path):
        print('Downloading GFPGANv1.3.pth (~340MB)...')
        urllib.request.urlretrieve(
            'https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.3.pth', path)
    return GFPGANer(model_path=path, upscale=scale, arch='clean',
                    channel_multiplier=2, bg_upsampler=upsampler)

print('Loader functions ready')

In [ ]:
# 5) Full Benchmark Suite

TEST_IMAGE = 'https://raw.githubusercontent.com/TencentARC/GFPGAN/master/inputs/whole_imgs/00.jpg'

# Download test image once
tmp_dir = tempfile.mkdtemp()
in_path = os.path.join(tmp_dir, 'in.jpg')
urllib.request.urlretrieve(TEST_IMAGE, in_path)
img_original = cv2.imread(in_path, cv2.IMREAD_COLOR)
h, w = img_original.shape[:2]
print(f'Test image size: {w}x{h}')

def run_benchmark(model_key, use_half=True, face_enhance=False, num_runs=3):
    """Run benchmark for a single model configuration"""
    print(f'\n--- Benchmark: {model_key} half={use_half} face={face_enhance} ---')
    
    # Load model
    if model_key == 'v3':
        upsampler, scale = get_v3_upsampler(use_half)
    else:
        upsampler, scale = get_rrdb_upsampler(model_key, use_half)
    
    # Face enhancer (if requested)
    face_enhancer = None
    if face_enhance:
        face_enhancer = get_face_enhancer(upsampler, scale)
    
    # Warm-up
    _ = upsampler.enhance(img_original, outscale=scale)
    
    # Benchmark
    times = []
    for i in range(num_runs):
        img = img_original.copy()
        t0 = time.time()
        if face_enhancer:
            _, _, output = face_enhancer.enhance(img, has_aligned=False,
                                                  only_center_face=False, paste_back=True)
        else:
            output, _ = upsampler.enhance(img, outscale=scale)
        t = time.time() - t0
        times.append(t)
        print(f'  Run {i+1}: {t:.3f}s')
    
    oh, ow = output.shape[:2]
    avg = sum(times) / len(times)
    print(f'  >>> Average: {avg:.3f}s | Output: {ow}x{oh}')
    return {'model': model_key, 'scale': scale, 'half': use_half,
            'face_enhance': face_enhance, 'avg_time': avg,
            'times': times, 'output_size': f'{ow}x{oh}'}

print('Benchmark function ready')

In [ ]:
# 6) Run benchmarks — all models + all configurations

results = []

# Standard models (FP16)
for model_key in ['x2plus', 'x4plus', 'anime', 'ultrasharp', 'v3']:
    r = run_benchmark(model_key, use_half=True, face_enhance=False, num_runs=3)
    results.append(r)

# FP32 comparison (for key models)
for model_key in ['x4plus', 'v3']:
    r = run_benchmark(model_key, use_half=False, face_enhance=False, num_runs=2)
    results.append(r)

# Face Enhance (GFPGAN) — x4plus + v3
for model_key in ['x4plus', 'v3']:
    r = run_benchmark(model_key, use_half=True, face_enhance=True, num_runs=2)
    results.append(r)

print('\n' + '='*60)
print('ALL BENCHMARK COMPLETE')
print('='*60)

In [ ]:
# 7) Results table

print(f'\n{"Model":<20} {"Scale":<6} {"FP16":<6} {"Face":<6} {"Avg Time":<10} {"Speed vs x4plus":<18}')
print('-'*70)

# Find x4plus FP16 baseline
baseline = None
for r in results:
    if r['model'] == 'x4plus' and r['half'] == True and r['face_enhance'] == False:
        baseline = r['avg_time']
        break

for r in results:
    model_display = r['model']
    scale = r['scale']
    half = 'FP16' if r['half'] else 'FP32'
    face = 'Yes' if r['face_enhance'] else 'No'
    avg = r['avg_time']
    
    if baseline and r['half'] and not r['face_enhance']:
        speedup = baseline / avg
        speed_str = f'{speedup:.2f}x {"faster" if speedup > 1 else "slower"}'
    else:
        speed_str = '-'
    
    print(f'{model_display:<20} {scale:<6} {half:<6} {face:<6} {avg:<10.3f}s {speed_str:<18}')

print(f'\nBaseline: x4plus FP16 = {baseline:.3f}s')
print(f'Test image: original size {w}x{h}')

In [ ]:
# 8) Visual comparison — pick 2 models to compare side-by-side

# Compare x4plus vs v3
compare_models = ['x4plus', 'v3']
comparison_results = []

for model_key in compare_models:
    print(f'Generating output for {model_key}...')
    if model_key == 'v3':
        upsampler, scale = get_v3_upsampler(use_half=True)
    else:
        upsampler, scale = get_rrdb_upsampler(model_key, use_half=True)
    output, _ = upsampler.enhance(img_original.copy(), outscale=scale)
    out_path = os.path.join(tmp_dir, f'out_{model_key}.png')
    cv2.imwrite(out_path, output)
    comparison_results.append((model_key, Image.open(out_path)))

# Display
fig, axes = plt.subplots(1, len(comparison_results) + 1, figsize=(20, 10))

# Original
img_orig = Image.open(in_path)
axes[0].imshow(img_orig)
axes[0].set_title(f'Original\n{img_orig.size[0]}x{img_orig.size[1]}', fontsize=12)
axes[0].axis('off')

for i, (name, img) in enumerate(comparison_results):
    axes[i+1].imshow(img)
    axes[i+1].set_title(f'{name}\n{img.size[0]}x{img.size[1]}', fontsize=12)
    axes[i+1].axis('off')

plt.tight_layout()
plt.show()
print('Done!')

In [ ]:
# 9) Manual test — adjust these vars and run

MODEL = 'x4plus'   # x2plus, x4plus, ultrasharp, anime, v3
FACE_ENHANCE = False
USE_HALF = True     # True = FP16, False = FP32

if MODEL == 'v3':
    upsampler, scale = get_v3_upsampler(USE_HALF)
else:
    upsampler, scale = get_rrdb_upsampler(MODEL, USE_HALF)

img = cv2.imread(in_path, cv2.IMREAD_COLOR)

if FACE_ENHANCE:
    face_enhancer = get_face_enhancer(upsampler, scale)
    _, _, output = face_enhancer.enhance(img, has_aligned=False,
                                         only_center_face=False, paste_back=True)
else:
    output, _ = upsampler.enhance(img, outscale=scale)

out_path = os.path.join(tmp_dir, 'out_manual.png')
cv2.imwrite(out_path, output)

img_orig = Image.open(in_path)
img_out = Image.open(out_path)

fig, axes = plt.subplots(1, 2, figsize=(16, 9))
axes[0].imshow(img_orig)
axes[0].set_title(f'Before\n{img_orig.size[0]}x{img_orig.size[1]}', fontsize=12)
axes[0].axis('off')
axes[1].imshow(img_out)
axes[1].set_title(f'After x{scale} ({MODEL})\n{img_out.size[0]}x{img_out.size[1]}', fontsize=12)
axes[1].axis('off')
plt.tight_layout()
plt.show()
print('Done!')